# بازتولید نسخه بازبینی‌شده Aave
این نوت‌بوک خروجی ساختگی ندارد. نتایج اجراهای انجام‌شده در بسته موجودند؛ سلول‌ها برای اجرای دوباره هستند.

فایل ZIP بازتولید را در Colab باز کنید و مسیر پروژه را در سلول بعدی قرار دهید. اعتبارسنجی آن‌چین نیازمند دسترسی آرشیوی واقعی است و در نبود آن، اجراشده تلقی نمی‌شود.

In [ ]:
from pathlib import Path
PROJECT = Path("/content/defi-systemic-risk")
assert (PROJECT / "revision").exists(), "ابتدا بسته را استخراج و مسیر پروژه را تنظیم کنید"
%cd {PROJECT}

In [ ]:
%pip install -r revision/requirements-lock.txt
%pip install --no-deps -e .

## محاسبه مجدد نتایج ذخیره‌شده
این مرحله به داده خام بزرگ نیاز ندارد و آموزش مجدد مدل‌ها نیست.

In [ ]:
!OPENBLAS_NUM_THREADS=1 python revision/scripts/audit_results.py
!OPENBLAS_NUM_THREADS=1 python revision/scripts/audit_remaining_tables.py
!python -m unittest discover -s tests -v

## دریافت منابع اصلی و بازآموزی واقعی
آرشیو گراف حدود ۱٫۱۳ گیگابایت است. فایل‌ها با هش منبع کنترل می‌شوند.

In [ ]:
!python -m defi_risk fetch
!python -m defi_risk prepare
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python -m defi_risk evaluate --out revision/results/reproduction

In [ ]:
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python revision/scripts/transfer_experiment.py
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python revision/scripts/later_holdout.py

## نمودارها و نسخه مقاله
ساخت PDF انگلیسی به Pandoc و TeX Live کامل نیاز دارد. PDF آماده در بسته موجود است.

In [ ]:
!python revision/scripts/make_figures.py
!python revision/scripts/make_review_report.py
# در صورت نصب ابزارهای LaTeX:
# !python paper/build_manuscript.py

## اتصال آرشیوی اختیاری
کلید را در Colab Secrets با نام ETH_ARCHIVE_RPC و ترجیحاً ETH_VALIDATION_RPC قرار دهید. در کد یا خروجی چاپ نشود. بررسی چند حساب، پنل کامل ریسک و اعتبارسنجی تمام Compound را ثابت نمی‌کند.

In [ ]:
import os
from google.colab import userdata
for name in ["ETH_ARCHIVE_RPC", "ETH_VALIDATION_RPC"]:
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        pass
if os.environ.get("ETH_ARCHIVE_RPC"):
    import subprocess
    subprocess.run(["python", "revision/scripts/historical_validation.py", "health", "--accounts", "revision/data/hf_spotcheck_requests.csv"], check=True)
else:
    print("دسترسی آرشیوی تنظیم نشده؛ هیچ اعتبارسنجی تاریخی اجرا نشد.")